# 04 - Indexing Artifacts Check and Build Commands

هذا الـ Notebook يختبر وجود ملفات الخرج الناتجة عن خدمات الفهرسة:
- SQLite document store
- Terrier index الخاص بـ TF-IDF/BM25
- BERT FAISS index
- Word2Vec FAISS index

إذا لم تكن الملفات موجودة على جهازك، لا مشكلة. هذا الـ Notebook يطبع أوامر البناء التي يجب أن يشغلها صديقك عنده على جهازه الذي يحتوي البيانات الكاملة.

In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
def file_size_mb(path):
    path = Path(path)
    if not path.exists() or path.is_dir():
        return None
    return round(path.stat().st_size / (1024 * 1024), 2)


def artifact_status_df():
    rows = []
    checks = {
        "SQLite document store": DB_PATH,
        "Terrier index folder (TF-IDF/BM25)": TERRIER_INDEX_PATH,
        "BERT index folder": BERT_INDEX_DIR,
        "BERT FAISS index": BERT_INDEX_DIR / "index.faiss",
        "BERT doc_ids.pkl": BERT_INDEX_DIR / "doc_ids.pkl",
        "BERT doc_ids.jsonl": BERT_INDEX_DIR / "doc_ids.jsonl",
        "BERT metadata.json": BERT_INDEX_DIR / "metadata.json",
        "Word2Vec index folder": WORD2VEC_INDEX_DIR,
        "Word2Vec FAISS index": WORD2VEC_INDEX_DIR / "index.faiss",
        "Word2Vec model": WORD2VEC_INDEX_DIR / "word2vec.model",
        "Word2Vec doc_ids.pkl": WORD2VEC_INDEX_DIR / "doc_ids.pkl",
        "Word2Vec metadata.json": WORD2VEC_INDEX_DIR / "metadata.json",
    }
    for name, path in checks.items():
        rows.append({"artifact": name, "path": str(path), "exists": Path(path).exists(), "size_MB": file_size_mb(path)})
    return pd.DataFrame(rows)

artifact_status_df()

In [ ]:
# قراءة metadata الخاصة بفهرس BERT وWord2Vec إن وجدت.
for p in [BERT_INDEX_DIR / "metadata.json", WORD2VEC_INDEX_DIR / "metadata.json"]:
    print("="*90)
    print("Metadata file:", p)
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
    else:
        print("Not found.")

In [ ]:
import pickle

def count_doc_ids(index_dir):
    index_dir = Path(index_dir)
    pkl_path = index_dir / "doc_ids.pkl"
    jsonl_path = index_dir / "doc_ids.jsonl"
    if pkl_path.exists():
        with open(pkl_path, "rb") as f:
            doc_ids = pickle.load(f)
        return len(doc_ids), doc_ids[:5]
    if jsonl_path.exists():
        sample = []
        count = 0
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if count < 5:
                    sample.append(line.strip())
                count += 1
        return count, sample
    return None, []

for name, d in [("BERT", BERT_INDEX_DIR), ("Word2Vec", WORD2VEC_INDEX_DIR)]:
    count, sample = count_doc_ids(d)
    print(name, "doc_ids count:", count)
    print(name, "sample:", sample)

In [ ]:
print("Commands to build missing artifacts from project scripts:")
print()
print("1) Build SQLite document store:")
print("python scripts/build_document_store.py --db-path data/documents.sqlite --batch-size 10000")
print()
print("2) Build Terrier index for TF-IDF/BM25:")
print("python scripts/build_terrier_index.py --db-path data/documents.sqlite --index-dir indexes/terrier_medline")
print()
print("3) Build BERT FAISS index:")
print("python scripts/build_bert_index.py --db-path data/documents.sqlite --index-dir indexes/faiss_bert --limit 10000 --batch-size 64")
print()
print("4) Build Word2Vec FAISS index:")
print("python scripts/build_word2vec_index.py --db-path data/documents.sqlite --index-dir indexes/faiss_word2vec --limit 10000")
print()
print("Important: remove --limit for the final full run if resources allow it.")